# GEMS Geological Layer Lab — launch in Colab

This is a **Streamlit app**, not a scientific notebook. Upload the `gems_layer_lab.zip` package, mount Drive, install dependencies, start Streamlit on port 8501, and open the served window.

The Streamlit process runs inside this Colab runtime and reads the existing GEMS_2 files from your mounted Drive. Keep the runtime connected while using the interface.


In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive/GEMS_2/data/raw/geodawn/gems-geodawn-numerical-features.tif').is_file(), 'Cannot find the GEMS_2 feature TIFF'
print('GEMS_2 source raster detected')


In [ ]:
# Get the app from the repository (pin a commit for reproducibility); a zip upload still works as a fallback.
import subprocess
from pathlib import Path
REF = 'main'                                   # or a pushed commit SHA
CODE = Path('/content/GEMS')
if not (CODE / '.git').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mybenkhadda/GEMS---drivendata.git', str(CODE)], check=True)
subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', ('origin/' + REF) if len(REF) < 40 else REF], check=True)
APP = CODE / 'streamlit' / 'gems_layer_lab'
print('app:', APP, '| commit', subprocess.run(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q -r /content/GEMS/streamlit/gems_layer_lab/requirements.txt

In [ ]:
import os, subprocess, time
os.environ['GEMS_ROOT'] = '/content/drive/MyDrive/GEMS_2'
log = open('/content/streamlit_feature_lab.log', 'w')
server = subprocess.Popen(['python', '-m', 'streamlit', 'run', 'app.py', '--server.port=8501', '--server.address=0.0.0.0',
                           '--server.headless=true'], cwd=str(APP), env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)
time.sleep(5)
assert server.poll() is None, 'Streamlit exited; inspect /content/streamlit_feature_lab.log'
print('Streamlit PID:', server.pid)

In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)


## Notes

* Everything you save (feature registry, tables, figures, tournaments, transects, GeoTIFF exports, manifests) is written to `/content/drive/MyDrive/GEMS_2/outputs/geological_layer_lab/` and survives runtime restarts. The original GEMS TIFFs are never written.
* Reload in a later session: re-run this notebook; the registry is read from Drive and saved features reappear (pages 2 / 7).
* Use saved features from P6 / M9 without Streamlit: `sys.path.insert(0, '/content/GEMS/streamlit/gems_layer_lab'); from lab.integration import FeatureStore`.
* Tests (synthetic, small): `!cd /content/GEMS/streamlit/gems_layer_lab && python -m pytest -q tests`.
* If the served window is unavailable, use the local launcher (`start_windows.bat` / `start_linux_mac.sh`) with locally synced TIFFs; do not expose a public unrestricted tunnel to private rasters.